# Temporary: Top-40 Tokens Per Sector

This notebook prints top-40 next-token probabilities for one representative report in each sector.
Use it to refine `sector_verbolizer` tokens.

In [1]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / 'src' / 'data_collection' / 'consts.py').is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import torch
from transformers import AutoModelForCausalLM

from src.data_collection.consts import DB_PARAMS
from src.data_collection.model_driver.model_driver_class import ModelDriver
from src.data_analysis.data_fetcher.data_fetcher_class import DataFetcher

/home/maxim-shibanov/anaconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Keep this prompt in sync with the one you are currently testing.
prompt = 'I think this company belongs to a sector of:'
top_k_tokens = 40
random_state = 42

# Current verbalizer baseline.
sector_verbolizer = {
    'Technology': ['technology', 'tech', 'software', 'semiconductor'],
    'Healthcare': ['healthcare', 'medical', 'pharma', 'biotech'],
    'Financial Services': ['financial', 'banking', 'finance', 'fin'],
    'Consumer Cyclical': ['cyclical', 'consumer', 'retail', 'auto'],
    'Consumer Defensive': ['defensive', 'staples', 'food', 'beverage'],
    'Industrials': ['industrials', 'industrial', 'indust', 'manufacturing'],
    'Energy': ['energy', 'oil', 'gas', 'natural'],
    'Utilities': ['utilities', 'utility', 'util', 'electric'],
    'Real Estate': ['real', 'estate', 'property', 'properties', 'reit'],
    'Basic Materials': ['materials', 'material', 'mining', 'basic'],
    'Communication Services': ['entertainment', 'media', 'communications', 'commun', 'internet'],
}

print('Sectors:', ', '.join(sector_verbolizer.keys()))

Sectors: Technology, Healthcare, Financial Services, Consumer Cyclical, Consumer Defensive, Industrials, Energy, Utilities, Real Estate, Basic Materials, Communication Services


In [3]:
model_name = 'mistralai/Mistral-7B-v0.1'
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    max_memory={0: '12GiB', 'cpu': '64GiB'},
)

model_driver = ModelDriver(model_name=model_name, model=model)

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.28s/it]
2026-05-29 11:58:53,629 [WARNING] Some parameters are on the meta device because they were offloaded to the cpu.


In [4]:
fetcher = DataFetcher(DB_PARAMS)
reports_df = fetcher._fetch_reports(regressors=['raw_text'])
reports_df = fetcher._apply_company_filters(reports_df, {})
reports_df = reports_df[reports_df['sector'].isin(sector_verbolizer.keys())].copy()
reports_df = reports_df[reports_df['raw_text'].notna()].copy()

sample_parts = []
for sector in sector_verbolizer:
    part = reports_df[reports_df['sector'] == sector].sample(n=1, random_state=random_state)
    sample_parts.append(part)
sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.sort_values('sector').reset_index(drop=True)

display(sample_df[['id', 'sector', 'ticker']])

/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:130: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2

Available regressors:
 - avg_default_verbolizer
 - avg_shrink_verbolizer
 - doc_len
 - eps_surprise
 - f_size
 - full_list_default_verbolizer
 - full_list_shrink_verbolizer
 - hv_orig_score
 - lm_orig_score
 - max_abs_default
 - max_abs_shrink
 - max_default_verbolizer
 - max_shrink_verbolizer
 - md_hv1
 - md_hv2
 - md_hv3
 - md_lm1
 - md_lm2
 - md_lm3
 - min_default_verbolizer
 - min_shrink_verbolizer
 - stretch_default
 - stretch_shrink
Available sectors:
 - Technology (92)
 - Industrials (86)
 - Financial Services (85)
 - Healthcare (66)
 - Consumer Cyclical (58)
 - Consumer Defensive (40)
 - Real Estate (32)
 - Utilities (32)
 - Energy (30)
 - Basic Materials (23)
 - Communication Services (22)


/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:169: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  companies_df = pd.read_sql_query(query, conn)


,id,sector,ticker
0,78013,Basic Materials,STLD
1,65254,Communication Services,VZ
2,76523,Consumer Cyclical,HAS
3,16559,Consumer Defensive,KR
4,66755,Energy,WMB
5,153529,Financial Services,COF
6,82404,Healthcare,COO
7,108723,Industrials,UPS
8,114546,Real Estate,MAA
9,139658,Technology,CSCO


In [5]:
def get_top_tokens(text: str, prompt: str, k: int = 40) -> list[tuple[str, float]]:
    clean_text = model_driver.clean_report(text)

    prompt_tokens = model_driver.tokenizer(prompt, return_tensors='pt')['input_ids'].squeeze()
    if prompt_tokens.dim() == 0:
        prompt_tokens = prompt_tokens.unsqueeze(0)

    context_max_tokens = 8000
    max_report_tokens = max(1, context_max_tokens - prompt_tokens.shape[0])
    report_tokens = model_driver.tokenizer(
        clean_text,
        return_tensors='pt',
        truncation=True,
        max_length=max_report_tokens,
    )['input_ids'].squeeze()
    if report_tokens.dim() == 0:
        report_tokens = report_tokens.unsqueeze(0)

    tokens = torch.cat((report_tokens, prompt_tokens), dim=0).to(model_driver.device)
    token_prob_dict = model_driver.fast_inference(tokens)
    return sorted(token_prob_dict.items(), key=lambda x: x[1], reverse=True)[:k]

In [6]:
top_tokens_by_sector = {}

for row in sample_df.itertuples(index=False):
    sector = row.sector
    top_tokens = get_top_tokens(row.raw_text, prompt, k=top_k_tokens)
    top_tokens_by_sector[sector] = top_tokens

    print(f'\n{sector}')
    print(f'Report id: {row.id} | ticker: {row.ticker}')
    print(f'Top {top_k_tokens} tokens by probability:')
    for token, prob in top_tokens:
        print(f'{token:<14} | {prob:.6f}')


Basic Materials
Report id: 78013 | ticker: STLD
Top 40 tokens by probability:
steel          | 0.057373
manufacturing  | 0.021118
Steel          | 0.019897
education      | 0.012451
it             | 0.010986
one            | 0.006256
industry       | 0.006042
our            | 0.005676
Manufact       | 0.005524
metal          | 0.005341
Education      | 0.004852
industrial     | 0.004730
companies      | 0.004303
very           | 0.003784
if             | 0.003677
heavy          | 0.003555
STE            | 0.003448
those          | 0.003143
engineering    | 0.003143
Industry       | 0.002853
Ste            | 0.002686
in             | 0.002609
highly         | 0.002441
The            | 0.002365
Construction   | 0.002365
Met            | 0.002228
the            | 0.002167
really         | 0.002090
employees      | 0.002029
great          | 0.001968
an             | 0.001907
talent         | 0.001907
Industrial     | 0.001907
all            | 0.001846
young          | 0.001678
mining     

In [7]:
# Optional: convert one sector output to DataFrame for easier copy/paste
sector_name = 'Real Estate'
pd.DataFrame(top_tokens_by_sector[sector_name], columns=['token', 'probability']).head(40)

,token,probability
0,RE,0.054199
1,Financial,0.042236
2,financial,0.020508
3,Finance,0.018066
4,Indust,0.016479
5,Property,0.011353
6,Energy,0.010986
7,https,0.010681
8,Commercial,0.010681
9,Real,0.008057
